# Machine Learning with Scikit-learn
## Supervised · Unsupervised · Evaluation · Pipelines · Tuning

---

## Table of Contents

**Foundations**
1. The Sklearn API and Train/Test Split
2. Linear Models — Regression and Classification
3. Decision Trees and Ensemble Methods
4. Support Vector Machines
5. K-Nearest Neighbors

**Model Evaluation**
6. Classification Metrics — Confusion Matrix, ROC, PR Curve
7. Regression Metrics and Cross-Validation

**Advanced**
8. Gradient Boosting — XGBoost and LightGBM
9. Unsupervised Learning — Clustering
10. Pipelines and Hyperparameter Tuning

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import (
    make_classification, make_regression, make_moons,
    load_breast_cancer, load_diabetes, load_wine
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)

# Shared binary classification dataset
X_clf, y_clf = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, class_sep=0.8, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Shared regression dataset
X_reg, y_reg = make_regression(n_samples=1000, n_features=15, n_informative=8, noise=30, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_tr_rs = scaler.fit_transform(X_tr_r)
X_te_rs  = scaler.transform(X_te_r)

print('Classification dataset:', X_clf.shape, '| classes:', np.bincount(y_clf))
print('Regression dataset:    ', X_reg.shape)

# Section 1 — The Sklearn API and Train/Test Split

## Concept

Scikit-learn follows a consistent, minimal API:
1. **Instantiate**: `model = SomeModel(hyperparams)`
2. **Fit**: `model.fit(X_train, y_train)`
3. **Predict**: `y_pred = model.predict(X_test)`
4. **Score**: `model.score(X_test, y_test)` or use `metrics.*`

**Transformers** additionally expose `transform` and `fit_transform`.

## Technical Deep Dive

**The golden rule of ML:** Never let test data influence training.

| Split strategy | Use case |
|---|---|
| `train_test_split` | Quick baseline evaluation |
| `KFold` / `StratifiedKFold` | Cross-validation for reliable estimates |
| `TimeSeriesSplit` | Time series data (no future leakage) |
| `GroupKFold` | Grouped data (patients, users) |

**Data leakage** = fitting preprocessors on the full dataset before splitting — the most common source of overly optimistic results.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Standard sklearn workflow
model = LogisticRegression(random_state=42, max_iter=500)
model.fit(X_tr_s, y_tr)

y_pred = model.predict(X_te_s)
y_prob = model.predict_proba(X_te_s)[:, 1]

print('Accuracy: ', accuracy_score(y_te, y_pred).round(4))
print('ROC-AUC:  ', roc_auc_score(y_te, y_prob).round(4))
print('F1 score: ', f1_score(y_te, y_pred).round(4))

# Cross-validation — more reliable than single split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(model, X_tr_s, y_tr, cv=cv,
                             scoring=['accuracy','roc_auc','f1'],
                             return_train_score=True)

print('\n5-Fold CV results:')
for metric in ['test_accuracy', 'test_roc_auc', 'test_f1']:
    scores = cv_results[metric]
    print(f'  {metric:20s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Demonstrating data leakage danger
from sklearn.feature_selection import SelectKBest, f_classif

# WRONG: fit selector on all data before splitting
selector_wrong = SelectKBest(f_classif, k=5)
X_selected_all = selector_wrong.fit_transform(X_clf, y_clf)  # leakage!
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_selected_all, y_clf, test_size=0.2, random_state=42)
model_wrong = LogisticRegression(max_iter=500).fit(X_tr_w, y_tr_w)
acc_wrong = model_wrong.score(X_te_w, y_te_w)

# CORRECT: fit selector only on training data
selector_right = SelectKBest(f_classif, k=5)
X_tr_orig, X_te_orig, y_tr_orig, y_te_orig = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
X_tr_c = selector_right.fit_transform(X_tr_orig, y_tr_orig)  # fit only on train
X_te_c = selector_right.transform(X_te_orig)                  # transform test
model_right = LogisticRegression(max_iter=500).fit(X_tr_c, y_tr_orig)
acc_right = model_right.score(X_te_c, y_te_orig)

print(f'Accuracy with leakage:    {acc_wrong:.4f}  (optimistically biased)')
print(f'Accuracy without leakage: {acc_right:.4f}  (honest estimate)')

## Summary

- Sklearn API: instantiate → fit → predict → evaluate.
- Always split data before fitting any preprocessor.
- Use cross-validation for reliable performance estimates.
- Use `Pipeline` to prevent leakage (Section 10).

---


# Section 2 — Linear Models

## Concept

Linear models are the foundation of ML: fast, interpretable, and often surprisingly effective.

**Regression:**
- **Linear Regression:** `y = Xw + b` — minimize MSE.
- **Ridge (L2):** adds `α||w||²` — shrinks all coefficients.
- **Lasso (L1):** adds `α||w||₁` — forces some coefficients to zero (feature selection).
- **ElasticNet:** combination of L1 and L2.

**Classification:**
- **Logistic Regression:** sigmoid of linear combination, outputs probability.
- **SGDClassifier:** stochastic gradient descent — scales to huge datasets.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

models_reg = {
    'Linear':    LinearRegression(),
    'Ridge':     Ridge(alpha=1.0),
    'Lasso':     Lasso(alpha=1.0, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000)
}

results = []
for name, model in models_reg.items():
    model.fit(X_tr_rs, y_tr_r)
    y_pred = model.predict(X_te_rs)
    results.append({
        'model': name,
        'RMSE':  np.sqrt(mean_squared_error(y_te_r, y_pred)).round(2),
        'MAE':   mean_absolute_error(y_te_r, y_pred).round(2),
        'R²':    r2_score(y_te_r, y_pred).round(4)
    })

print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# Regularization effect on coefficients
alphas = [0.001, 0.1, 1, 10, 100, 1000]
ridge_coefs = [Ridge(alpha=a).fit(X_tr_rs, y_tr_r).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_tr_rs, y_tr_r).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for coefs, title, ax in zip([ridge_coefs, lasso_coefs], ['Ridge (L2)', 'Lasso (L1)'], axes):
    for i, coef in enumerate(np.array(coefs).T):
        ax.semilogx(alphas, coef, alpha=0.6)
    ax.set_xlabel('Alpha (regularization strength)')
    ax.set_ylabel('Coefficient value')
    ax.set_title(f'{title} — Coefficient Paths')
    ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic regression with regularization types
models_clf = {
    'LR (L2)':  LogisticRegression(C=1.0, penalty='l2', max_iter=500, random_state=42),
    'LR (L1)':  LogisticRegression(C=1.0, penalty='l1', solver='saga', max_iter=500, random_state=42),
    'LR (None)': LogisticRegression(penalty=None, max_iter=500, random_state=42)
}

for name, model in models_clf.items():
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)
    y_prob = model.predict_proba(X_te_s)[:, 1]
    n_nonzero = (model.coef_[0] != 0).sum()
    print(f'{name:12s} | Acc={accuracy_score(y_te,y_pred):.4f} | AUC={roc_auc_score(y_te,y_prob):.4f} | Non-zero coefs: {n_nonzero}/{X_tr_s.shape[1]}')

## Exercises

1. Use `RidgeCV` to automatically select the best alpha via cross-validation.
2. Fit a Lasso model and print which features it zeroed out.
3. Compare logistic regression with C=0.01, 0.1, 1, 10 on accuracy and AUC.
4. Implement polynomial features (degree=2) + Ridge regression on the regression dataset.

## Summary

- Ridge: shrinks all coefficients — good default for regression.
- Lasso: zeros out weak features — built-in feature selection.
- Logistic Regression: `C = 1/α` (higher C = less regularization).
- Always scale features for linear models.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures

# Exercise 1: RidgeCV
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
ridge_cv.fit(X_tr_rs, y_tr_r)
print(f'Best Ridge alpha: {ridge_cv.alpha_:.4f}')
y_pred_rcv = ridge_cv.predict(X_te_rs)
print(f'Test R²: {r2_score(y_te_r, y_pred_rcv):.4f}')

# Exercise 2: Lasso zero features
lasso = Lasso(alpha=2.0, max_iter=5000).fit(X_tr_rs, y_tr_r)
zero_features = np.where(lasso.coef_ == 0)[0]
print(f'\nLasso zeroed out features: {zero_features.tolist()}')

# Exercise 4: Polynomial + Ridge
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = poly.fit_transform(X_tr_rs[:, :5])  # use 5 features to keep matrix manageable
X_te_poly = poly.transform(X_te_rs[:, :5])
ridge_poly = Ridge(alpha=1.0).fit(X_tr_poly, y_tr_r)
print(f'\nPolynomial (deg=2) Ridge R²: {r2_score(y_te_r, ridge_poly.predict(X_te_poly)):.4f}')

# Section 3 — Decision Trees and Ensemble Methods

## Concept

**Decision trees** split features to minimize impurity (Gini, entropy, MSE).
They are interpretable but prone to overfitting.

**Ensembles** combine many trees to reduce variance (bagging) or bias (boosting):

| Method | Strategy | Weak Learners |
|---|---|---|
| Random Forest | Bagging + random features | Decision Trees |
| Extra Trees | More randomness | Decision Trees |
| AdaBoost | Reweight misclassified | Stumps |
| Gradient Boosting | Fit residuals | Trees |
| XGBoost / LightGBM / CatBoost | Optimized GB | Trees |

**Bias-Variance Trade-off:**
- Underfitting = high bias, low variance.
- Overfitting = low bias, high variance.
- Bagging reduces variance; Boosting reduces bias.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier

# Single tree — shows overfitting at high depth
print('=== Decision Tree depth comparison ===')
for depth in [2, 5, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_tr_s, y_tr)
    train_acc = dt.score(X_tr_s, y_tr)
    test_acc  = dt.score(X_te_s, y_te)
    print(f'depth={str(depth):4s} | train={train_acc:.4f} | test={test_acc:.4f} | {"overfit" if train_acc - test_acc > 0.05 else "ok"}')

In [ ]:
# Visualize a shallow decision tree
dt_viz = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_viz.fit(X_tr_s, y_tr)

fig, ax = plt.subplots(figsize=(14, 5))
plot_tree(dt_viz, max_depth=3, filled=True, feature_names=[f'F{i}' for i in range(X_tr_s.shape[1])],
          class_names=['0','1'], impurity=True, proportion=True, ax=ax)
ax.set_title('Decision Tree (depth=3)')
plt.tight_layout(); plt.show()

In [ ]:
# Ensemble comparison
ensembles = {
    'Random Forest (100)':     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees (100)':       ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting (100)': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results_ens = []
for name, clf in ensembles.items():
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    y_prob = clf.predict_proba(X_te_s)[:, 1]
    results_ens.append({'model': name,
                        'accuracy': accuracy_score(y_te, y_pred).round(4),
                        'f1':       f1_score(y_te, y_pred).round(4),
                        'roc_auc':  roc_auc_score(y_te, y_prob).round(4)})

print(pd.DataFrame(results_ens).to_string(index=False))

In [ ]:
# Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr_s, y_tr)

feat_names = [f'Feature_{i}' for i in range(X_tr_s.shape[1])]
importance = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
importance.plot.bar(ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Random Forest Feature Importances')
ax.set_ylabel('Mean Decrease in Impurity')
ax.set_xlabel('Feature')
plt.tight_layout(); plt.show()

## Exercises

1. Plot the learning curve for a Random Forest: accuracy vs number of training samples.
2. Tune `max_depth`, `min_samples_leaf` in a Decision Tree using `GridSearchCV`.
3. Compute permutation importance on the Random Forest test set.
4. Compare `n_estimators` from 10 to 500 on OOB score for Random Forest.

## Mini Challenge

Build a stacking ensemble: train 3 base models (LogisticRegression, RandomForest, GradientBoosting)
and use a Logistic Regression meta-learner on their out-of-fold predictions.
Compare to individual models.

## Summary

- Single trees overfit easily — control with `max_depth`, `min_samples_leaf`.
- Random Forest: parallel, low variance, high accuracy, great default.
- Gradient Boosting: sequential, lower bias, often best accuracy but slower.
- Feature importance: use for insight, but cross-validate before dropping features.

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
from sklearn.inspection import permutation_importance

# Exercise 3: Permutation importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_s, y_tr)
perm = permutation_importance(rf, X_te_s, y_te, n_repeats=10, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({
    'feature':   feat_names,
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print('Permutation Importance:')
print(perm_df.round(4))

# Exercise 4: OOB score vs n_estimators
oob_scores = []
for n in [10, 30, 50, 100, 200, 500]:
    rf_oob = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_oob.fit(X_tr_s, y_tr)
    oob_scores.append(rf_oob.oob_score_)

print('\nOOB scores:', dict(zip([10,30,50,100,200,500], [round(s,4) for s in oob_scores])))

In [ ]:
# Mini Challenge: Stacking Ensemble
# YOUR CODE HERE

# Section 4 — Support Vector Machines

## Concept

SVMs find the **maximum margin hyperplane** separating classes.
Support vectors are the data points closest to the decision boundary.

**Kernel trick:** maps data to higher-dimensional space to find a linear separator:
- `linear`: no transformation
- `rbf`: radial basis function (Gaussian) — most common
- `poly`: polynomial kernel
- `sigmoid`

**Key hyperparameters:**
- `C`: regularization — small C = wider margin (more misclassifications allowed)
- `gamma` (rbf): controls influence radius of each support vector


In [ ]:
from sklearn.svm import SVC, SVR

# SVM with different kernels
kernels = ['linear', 'rbf', 'poly']
for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, probability=True, random_state=42)
    svm.fit(X_tr_s, y_tr)
    y_pred = svm.predict(X_te_s)
    y_prob = svm.predict_proba(X_te_s)[:, 1]
    print(f'SVM kernel={kernel:6s} | Acc={accuracy_score(y_te,y_pred):.4f} | AUC={roc_auc_score(y_te,y_prob):.4f}')

In [ ]:
# Visualize decision boundary on 2D moons dataset
X_m, y_m = make_moons(n_samples=300, noise=0.2, random_state=42)
X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(X_m, y_m, test_size=0.25, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, kernel in zip(axes, ['linear','rbf','poly']):
    svm = SVC(kernel=kernel, C=2.0, gamma='scale').fit(X_m_tr, y_m_tr)

    # Decision boundary mesh
    xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-2, 2, 200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_m_te[:, 0], X_m_te[:, 1], c=y_m_te, cmap='coolwarm', edgecolors='white', s=40)
    acc = svm.score(X_m_te, y_m_te)
    ax.set_title(f'SVM kernel={kernel} (acc={acc:.3f})')

plt.tight_layout(); plt.show()

## Summary

- SVMs excel with high-dimensional, clear-margin data.
- RBF kernel is the best default for non-linear data.
- Scale features before SVM — mandatory.
- SVMs are slow on large datasets (`n > 50k`); use SGDClassifier instead.

---


# Section 5 — K-Nearest Neighbors

## Concept

KNN is a **lazy learner** — no explicit training phase.
To classify a new point, it finds the k nearest neighbors (by distance) and takes a majority vote.

**Distance metrics:** Euclidean, Manhattan, Minkowski, Cosine.

**Key hyperparameters:**
- `k`: small k = complex boundary (overfit); large k = smooth boundary (underfit).
- `weights`: `'uniform'` or `'distance'` (closer neighbors count more).


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# k vs accuracy
k_values = [1, 3, 5, 7, 10, 15, 20, 30]
train_accs, test_accs = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_s, y_tr)
    train_accs.append(knn.score(X_tr_s, y_tr))
    test_accs.append(knn.score(X_te_s, y_te))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, train_accs, 'o-', label='Train', color='steelblue')
ax.plot(k_values, test_accs, 's-', label='Test', color='tomato')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Accuracy vs k')
ax.legend()
ax.set_xticks(k_values)
plt.tight_layout(); plt.show()

best_k = k_values[np.argmax(test_accs)]
print(f'Best k={best_k} | Test accuracy={max(test_accs):.4f}')

## Summary

- KNN: no training, slow prediction (`O(n)` per query without kd-tree).
- Highly sensitive to scale — always standardize.
- Curse of dimensionality: KNN degrades in very high dimensions.
- Use as baseline or when decision boundaries are complex and data is small.

---


# Section 6 — Classification Metrics

## Concept

Accuracy alone is misleading for imbalanced datasets.
Choose metrics aligned with the business problem.

## Technical Deep Dive

| Metric | Formula | When to use |
|---|---|---|
| Accuracy | (TP+TN)/N | Balanced classes |
| Precision | TP/(TP+FP) | Cost of false positive is high |
| Recall | TP/(TP+FN) | Cost of false negative is high |
| F1 | 2*P*R/(P+R) | Balance precision/recall |
| F-beta | (1+β²)*P*R/(β²*P+R) | Weight recall over precision (β>1) |
| ROC-AUC | Area under ROC | Threshold-independent, balanced |
| PR-AUC | Area under PR | Highly imbalanced datasets |
| MCC | Matthews Correlation | Single balanced metric |

**Confusion matrix terminology:**
- TP: correctly predicted positive
- TN: correctly predicted negative
- FP: predicted positive, actually negative (Type I error)
- FN: predicted negative, actually positive (Type II error)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr_s, y_tr)
y_pred = rf.predict(X_te_s)
y_prob = rf.predict_proba(X_te_s)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_te, y_pred, target_names=['Class 0','Class 1']))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# ROC curve
RocCurveDisplay.from_predictions(y_te, y_prob, ax=axes[1], name='RF')
axes[1].set_title('ROC Curve')
axes[1].plot([0,1],[0,1],'--k', label='Random')
axes[1].legend()

# Precision-Recall curve
PrecisionRecallDisplay.from_predictions(y_te, y_prob, ax=axes[2], name='RF')
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score as f1

# Threshold tuning — different thresholds give different precision/recall tradeoffs
thresholds = np.arange(0.1, 0.91, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_te, y_t, zero_division=0))
    recalls.append(recall_score(y_te, y_t))
    f1s.append(f1(y_te, y_t, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precisions, label='Precision', linewidth=2)
ax.plot(thresholds, recalls, label='Recall', linewidth=2)
ax.plot(thresholds, f1s, label='F1', linewidth=2, linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='Default 0.5')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold')
ax.legend()
plt.tight_layout(); plt.show()

best_t = thresholds[np.argmax(f1s)]
print(f'Best threshold for F1: {best_t:.2f} → F1={max(f1s):.4f}')

## Exercises

1. Create an imbalanced dataset (5% positive) and compare accuracy vs PR-AUC.
2. Find the optimal threshold that maximizes F2 score (recall is twice as important as precision).
3. Plot ROC curves for 3 models on the same axes and compare AUCs.
4. Compute MCC (Matthews Correlation Coefficient) for the Random Forest predictions.

## Summary

- Never use accuracy alone for imbalanced data.
- F1: balanced precision/recall. PR-AUC: better than ROC-AUC for imbalanced.
- Tune threshold based on business cost: FP cost vs FN cost.
- Use `classification_report` as the first evaluation step.

---


### Exercise and Challenge Solutions — Section 6


In [ ]:
from sklearn.metrics import matthews_corrcoef, fbeta_score, average_precision_score

# Exercise 4: MCC
mcc = matthews_corrcoef(y_te, y_pred)
print(f'MCC: {mcc:.4f}')

# Exercise 1: imbalanced dataset
X_imb, y_imb = make_classification(n_samples=2000, weights=[0.95, 0.05], random_state=42)
X_ti, X_ei, y_ti, y_ei = train_test_split(X_imb, y_imb, test_size=0.2, stratify=y_imb, random_state=42)
X_ti_s = StandardScaler().fit_transform(X_ti)
X_ei_s = StandardScaler().fit_transform(X_ei)
rf_imb = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_ti_s, y_ti)
y_imb_pred = rf_imb.predict(X_ei_s)
y_imb_prob = rf_imb.predict_proba(X_ei_s)[:, 1]
print(f'\nImbalanced dataset: accuracy={accuracy_score(y_ei, y_imb_pred):.4f}  (misleading)')
print(f'PR-AUC (honest):    {average_precision_score(y_ei, y_imb_prob):.4f}')

# Exercise 2: F2 optimal threshold
f2s = [fbeta_score(y_te, (y_prob >= t).astype(int), beta=2, zero_division=0) for t in thresholds]
best_f2_t = thresholds[np.argmax(f2s)]
print(f'\nBest threshold for F2: {best_f2_t:.2f} → F2={max(f2s):.4f}')

# Section 7 — Regression Metrics and Cross-Validation

## Concept

**Regression metrics:**

| Metric | Formula | Interpretation |
|---|---|---|
| MAE | `mean(|y - ŷ|)` | Average absolute error, same units |
| MSE | `mean((y - ŷ)²)` | Penalizes large errors more |
| RMSE | `sqrt(MSE)` | Same units as target |
| R² | `1 - SS_res/SS_tot` | Proportion of variance explained |
| MAPE | `mean(|y - ŷ|/|y|)` | Percentage error (avoid if y≈0) |

**Cross-validation strategies:**
- `KFold`: standard, good for large datasets.
- `StratifiedKFold`: preserves class ratios — use for classification.
- `RepeatedKFold`: repeat CV with different random splits for more stable estimates.
- `TimeSeriesSplit`: walk-forward validation — never use future data.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

reg_models = {
    'Ridge':    Ridge(alpha=1.0),
    'RF Reg':   RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GB Reg':   GradientBoostingRegressor(n_estimators=100, random_state=42)
}

reg_results = []
for name, model in reg_models.items():
    model.fit(X_tr_rs, y_tr_r)
    y_pred = model.predict(X_te_rs)
    reg_results.append({
        'model': name,
        'MAE':   mean_absolute_error(y_te_r, y_pred).round(2),
        'RMSE':  np.sqrt(mean_squared_error(y_te_r, y_pred)).round(2),
        'R²':    r2_score(y_te_r, y_pred).round(4)
    })

print(pd.DataFrame(reg_results).to_string(index=False))

# Residual plot for best model
best_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
best_reg.fit(X_tr_rs, y_tr_r)
y_pred_best = best_reg.predict(X_te_rs)
residuals = y_te_r - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5, s=25)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residual Plot (GBR)')

axes[1].scatter(y_te_r, y_pred_best, alpha=0.5, s=25, color='steelblue')
mn, mx = y_te_r.min(), y_te_r.max()
axes[1].plot([mn,mx],[mn,mx], 'r--', label='Perfect')
axes[1].set_xlabel('True'); axes[1].set_ylabel('Predicted')
axes[1].set_title(f'True vs Predicted (R²={r2_score(y_te_r,y_pred_best):.4f})')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.model_selection import KFold, RepeatedKFold, TimeSeriesSplit

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

cv_strategies = {
    'KFold(5)':          KFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedKFold(5x3)': RepeatedKFold(n_splits=5, n_repeats=3, random_state=42),
    'TimeSeriesSplit(5)': TimeSeriesSplit(n_splits=5)
}

print('Cross-validation R² scores:')
for cv_name, cv in cv_strategies.items():
    scores = cross_val_score(rf_reg, X_reg, y_reg, cv=cv, scoring='r2', n_jobs=-1)
    print(f'  {cv_name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

## Summary

- RMSE: good default — penalizes large errors. MAE: robust to outliers.
- R² = 1 means perfect prediction; R² = 0 means prediction is as good as predicting the mean.
- Residual plots: should be random around 0. Patterns = model misspecification.
- Use `RepeatedKFold` for robust estimates on small datasets.

---


# Section 8 — Gradient Boosting: XGBoost and LightGBM

## Concept

XGBoost and LightGBM are optimized gradient boosting libraries that dominate tabular ML competitions.
Both are significantly faster and more accurate than sklearn's `GradientBoostingClassifier`.

## Technical Deep Dive

| Feature | XGBoost | LightGBM |
|---|---|---|
| Tree growth | Level-wise | Leaf-wise (faster) |
| Speed | Fast | Faster (10x+ on large data) |
| Memory | Moderate | Less |
| Categorical support | Manual encoding | Native |
| Missing values | Native | Native |
| Main params | `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree` | Same + `num_leaves` |

**Key technique: early stopping** — stop training when validation metric stops improving.


In [ ]:
try:
    import xgboost as xgb

    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        early_stopping_rounds=30,
        random_state=42,
        verbosity=0
    )

    xgb_model.fit(X_tr_s, y_tr,
                  eval_set=[(X_te_s, y_te)],
                  verbose=False)

    y_pred_xgb = xgb_model.predict(X_te_s)
    y_prob_xgb = xgb_model.predict_proba(X_te_s)[:, 1]

    print(f'XGBoost best iteration: {xgb_model.best_iteration}')
    print(f'Accuracy: {accuracy_score(y_te, y_pred_xgb):.4f}')
    print(f'ROC-AUC:  {roc_auc_score(y_te, y_prob_xgb):.4f}')

except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')
    print('XGBoost: n_estimators=500, early stopping, max_depth=4, lr=0.05')

In [ ]:
try:
    import lightgbm as lgb

    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        random_state=42,
        verbose=-1
    )

    lgb_model.fit(
        X_tr_s, y_tr,
        eval_set=[(X_te_s, y_te)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    )

    y_pred_lgb = lgb_model.predict(X_te_s)
    y_prob_lgb = lgb_model.predict_proba(X_te_s)[:, 1]

    print(f'LightGBM best iteration: {lgb_model.best_iteration_}')
    print(f'Accuracy: {accuracy_score(y_te, y_pred_lgb):.4f}')
    print(f'ROC-AUC:  {roc_auc_score(y_te, y_prob_lgb):.4f}')

except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')
    print('LightGBM: leaf-wise growth, very fast on large datasets')

## Summary

- XGBoost and LightGBM beat sklearn's GBM in speed and accuracy.
- Always use early stopping with a validation set — prevents overfitting automatically.
- LightGBM is preferred for large datasets (>100k rows).
- Key tuning: `learning_rate`, `num_leaves`/`max_depth`, `subsample`, `colsample_bytree`, `min_child_samples`.

---


# Section 9 — Unsupervised Learning: Clustering

## Concept

Clustering groups data points by similarity without labeled targets.

| Algorithm | Key Idea | Params | Strengths |
|---|---|---|---|
| K-Means | Minimize within-cluster variance | k | Fast, scalable |
| DBSCAN | Density-based connected regions | eps, min_samples | Arbitrary shape, finds outliers |
| Hierarchical | Nested merges (agglomerative) | n_clusters, linkage | No k needed, dendrogram |
| Gaussian Mixture | Probabilistic soft assignment | n_components | Soft clusters, elliptical |

**Evaluation (no ground truth):**
- Silhouette score: [-1, 1] — higher is better.
- Calinski-Harabasz: higher is better.
- Inertia (K-Means): lower is better.
- Elbow method: plot inertia vs k.


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# Use PCA 2D embedding for visualization
from sklearn.decomposition import PCA
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(StandardScaler().fit_transform(X_clf))

# Elbow method for K-Means
inertias, silhouettes = [], []
ks = range(2, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_2d)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_2d, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].set_xticks(ks)

axes[1].plot(ks, silhouettes, 's-', color='tomato')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xticks(ks)

best_k = ks[np.argmax(silhouettes)]
print(f'Best k by silhouette: {best_k}')
plt.tight_layout(); plt.show()

In [ ]:
# Compare clustering algorithms
clusterers = {
    f'K-Means (k={best_k})': KMeans(n_clusters=best_k, random_state=42, n_init='auto'),
    'DBSCAN':                 DBSCAN(eps=0.5, min_samples=10),
    f'Agglomerative (k={best_k})': AgglomerativeClustering(n_clusters=best_k),
    f'GMM (k={best_k})':      GaussianMixture(n_components=best_k, random_state=42)
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, clf) in zip(axes, clusterers.items()):
    if hasattr(clf, 'fit_predict'):
        labels = clf.fit_predict(X_2d)
    else:  # GMM uses predict
        clf.fit(X_2d)
        labels = clf.predict(X_2d)

    n_found = len(set(labels)) - (1 if -1 in labels else 0)
    sil = silhouette_score(X_2d, labels) if len(set(labels)) > 1 and -1 not in labels else 'N/A (noise)'
    sil_str = f'{sil:.3f}' if isinstance(sil, float) else sil

    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='Set1', s=15, alpha=0.7)
    ax.set_title(f'{name}\nclusters={n_found}, sil={sil_str}')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

## Exercises

1. Apply K-Means on the `load_wine` dataset and compare cluster labels to true wine classes.
2. Use DBSCAN on `make_moons` data and visualize the clusters and noise points.
3. Build a hierarchical clustering dendrogram using `scipy.cluster.hierarchy`.
4. Determine optimal DBSCAN `eps` using a k-distance graph.

## Summary

- K-Means: fast, requires k upfront, assumes spherical clusters.
- DBSCAN: no k needed, handles arbitrary shapes, identifies noise points.
- Silhouette score: best single metric for cluster quality without labels.
- Elbow method + silhouette together for robust k selection.

---


# Section 10 — Pipelines and Hyperparameter Tuning

## Concept

A **Pipeline** chains preprocessing and model into a single object:
- Prevents data leakage.
- Makes code cleaner and production-ready.
- Enables end-to-end cross-validation and tuning.

**Hyperparameter tuning strategies:**

| Strategy | Description | Use When |
|---|---|---|
| `GridSearchCV` | Exhaustive search over param grid | Small param space |
| `RandomizedSearchCV` | Random samples from distributions | Large param space |
| `HalvingGridSearchCV` | Successive halving — fast | Medium param space |
| Optuna / Hyperopt | Bayesian optimization | Complex spaces |


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Real-world pipeline with mixed types (numeric + categorical)
# Load breast cancer dataset as example
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
feature_names = bc.feature_names

X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)

# All numeric in this dataset — simpler pipeline
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipeline = Pipeline([
    ('preprocess', numeric_pipe),
    ('clf',        RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit and evaluate the pipeline
pipeline.fit(X_bc_tr, y_bc_tr)
y_bc_pred = pipeline.predict(X_bc_te)
y_bc_prob = pipeline.predict_proba(X_bc_te)[:, 1]

print('Pipeline — Breast Cancer Classification:')
print(f'  Accuracy: {accuracy_score(y_bc_te, y_bc_pred):.4f}')
print(f'  ROC-AUC:  {roc_auc_score(y_bc_te, y_bc_prob):.4f}')
print(classification_report(y_bc_te, y_bc_pred, target_names=bc.target_names))

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# GridSearchCV
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth':    [None, 5, 10],
    'clf__min_samples_leaf': [1, 3, 5]
}

gs = GridSearchCV(
    pipeline, param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1, verbose=0
)
gs.fit(X_bc_tr, y_bc_tr)

print('GridSearchCV best params:')
print(gs.best_params_)
print(f'Best CV AUC: {gs.best_score_:.4f}')
print(f'Test AUC:    {roc_auc_score(y_bc_te, gs.predict_proba(X_bc_te)[:, 1]):.4f}')

In [ ]:
from scipy.stats import randint, uniform

# RandomizedSearchCV — better for large spaces
param_dist = {
    'clf__n_estimators':    randint(50, 500),
    'clf__max_depth':       [None, 3, 5, 10, 15, 20],
    'clf__min_samples_leaf': randint(1, 20),
    'clf__max_features':    uniform(0.3, 0.7)
}

rs = RandomizedSearchCV(
    pipeline, param_dist,
    n_iter=50, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=0
)
rs.fit(X_bc_tr, y_bc_tr)

print('RandomizedSearchCV best params:')
print(rs.best_params_)
print(f'Best CV AUC: {rs.best_score_:.4f}')
print(f'Test AUC:    {roc_auc_score(y_bc_te, rs.predict_proba(X_bc_te)[:, 1]):.4f}')

In [ ]:
# Full pipeline with mixed types (numeric + categorical)
from sklearn.compose import ColumnTransformer

# Simulate a mixed dataset
n_samples = 500
rng2 = np.random.default_rng(42)
df_mixed = pd.DataFrame({
    'age':       rng2.integers(20, 65, n_samples),
    'salary':    rng2.integers(30000, 150000, n_samples),
    'score':     rng2.normal(70, 15, n_samples),
    'dept':      rng2.choice(['Eng','HR','Sales','Mktg'], n_samples),
    'gender':    rng2.choice(['M','F'], n_samples),
    'target':    rng2.choice([0, 1], n_samples, p=[0.7, 0.3])
})
df_mixed.loc[rng2.choice(n_samples, 30, replace=False), 'score'] = np.nan

X_mixed = df_mixed.drop('target', axis=1)
y_mixed = df_mixed['target']

num_features = ['age', 'salary', 'score']
cat_features = ['dept', 'gender']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

full_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf',  RandomForestClassifier(n_estimators=100, random_state=42))
])

X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(X_mixed, y_mixed, test_size=0.2, random_state=42, stratify=y_mixed)
full_pipeline.fit(X_m_tr, y_m_tr)

print('Mixed-type Pipeline:')
print(f'  Accuracy: {accuracy_score(y_m_te, full_pipeline.predict(X_m_te)):.4f}')
print(f'  ROC-AUC:  {roc_auc_score(y_m_te, full_pipeline.predict_proba(X_m_te)[:,1]):.4f}')

## Exercises

1. Build a pipeline for the regression task: impute → scale → Lasso. Tune alpha with `GridSearchCV`.
2. Add a PCA step to the classification pipeline and tune `n_components` + `clf__n_estimators` together.
3. Use `HalvingGridSearchCV` (sklearn 0.24+) for faster tuning — compare time to standard grid search.
4. Implement a custom sklearn transformer (inheriting `BaseEstimator, TransformerMixin`) that log-transforms a specific column.

## Mini Challenge

Build a complete end-to-end ML system:
- Mixed-type ColumnTransformer
- Compare: LogisticRegression, RandomForest, GradientBoosting inside the pipeline
- Select the best model using 5-fold cross-validation on ROC-AUC
- Report final test metrics and feature importances

## Best Practices

- Always use `Pipeline` in production — eliminates entire class of bugs.
- Use `RandomizedSearchCV` first to identify good regions, then `GridSearchCV` to fine-tune.
- Parallelize search with `n_jobs=-1`.
- Report CV metrics with std — a model with slightly lower mean but much lower std is often better.

## Common Mistakes

- Fitting scaler on all data before splitting — causes data leakage.
- Using test set during hyperparameter tuning — use a separate validation set or CV.
- Treating cross-validation score as the final metric — always evaluate on held-out test set.
- Ignoring class imbalance — use `class_weight='balanced'` or SMOTE.

## Summary

- `Pipeline` = safe, clean, production-ready ML.
- `ColumnTransformer` handles mixed-type features.
- GridSearch for small spaces; RandomizedSearch for large spaces.
- Prefix pipeline step params with `step_name__param_name` in param grids.

---


### Exercise and Challenge Solutions — Section 10


In [ ]:
# Exercise 4: Custom transformer
# YOUR CODE HERE

In [ ]:
# Mini Challenge: full end-to-end ML system
# YOUR CODE HERE

# Course Summary

| Section | Key Skills |
|---|---|
| 1. Sklearn API | fit/predict, CV, data leakage prevention |
| 2. Linear Models | LinearReg, Ridge, Lasso, LogisticReg, regularization |
| 3. Trees & Ensembles | DT overfitting, RF, GB, feature importance, stacking |
| 4. SVM | Kernel trick, C, gamma, decision boundaries |
| 5. KNN | k selection, distance metrics, scaling |
| 6. Classification Metrics | Confusion matrix, ROC, PR curve, threshold tuning |
| 7. Regression Metrics & CV | RMSE/MAE/R², residual plots, K-Fold strategies |
| 8. XGBoost/LightGBM | Leaf-wise growth, early stopping, key params |
| 9. Clustering | K-Means, DBSCAN, Agglomerative, silhouette, elbow |
| 10. Pipelines & Tuning | ColumnTransformer, GridSearch, RandomSearch |

## Model Selection Guide

| Situation | Recommended Model |
|---|---|
| Interpretability needed | Logistic Regression, Decision Tree |
| Tabular data, accuracy first | LightGBM or XGBoost |
| High-dimensional sparse | Logistic Reg (L1), LinearSVC |
| Small dataset | RF, SVM, KNN |
| Very large dataset (>1M rows) | LightGBM, SGDClassifier |
| Baseline | Logistic Regression, Ridge/Lasso |

## Next Steps

- **`deep_learning_course.ipynb`** — Neural Networks, PyTorch, CNNs, Transformers

---
